# Chapter 15: Fine-Tuning — Adapting a Pre-Trained Model

Pre-training an LLM costs millions of dollars (Ch 13). But you can **adapt** it to your specific task cheaply.

```
Pre-trained model (general knowledge)
    ↓ fine-tune on medical texts
Medical expert model

Pre-trained model (general knowledge)
    ↓ fine-tune on legal documents
Legal expert model
```

Fine-tuning = continue training on a **small, specific dataset** while keeping most of what the model already knows.

---
## Why Not Train From Scratch?

| Approach | Data Needed | Cost | Time | Quality |
|----------|-------------|------|------|--------|
| Train from scratch | Trillions of tokens | $10M+ | Months | Best (if you have data) |
| Full fine-tune | 10K-100K examples | $100-1K | Hours | Great |
| LoRA fine-tune | 1K-10K examples | $10-100 | Minutes-hours | Good |
| Prompt engineering | 0 examples | $0 | Instant | Decent |

Fine-tuning is the sweet spot: much cheaper than pre-training, much better than just prompting.

In [ ]:
import numpy as np
np.random.seed(42)

# Visualize the cost-quality tradeoff
print("Cost vs Quality Tradeoff:\n")
print("  Quality")
print("  ↑")
print("  │          ★ Train from scratch ($10M+)")
print("  │       ★ Full fine-tune ($100-1K)")
print("  │     ★ LoRA fine-tune ($10-100)")
print("  │")
print("  │  ★ Prompt engineering ($0)")
print("  │")
print("  └──────────────────────────→ Cost")
print()
print("Key insight: fine-tuning gets 90% of the quality at 1% of the cost.")
print("The pre-trained model already knows language — you just teach it your task.")

---
## Full Fine-Tuning: Update All Parameters

The simplest approach: continue training the entire model on your data.

```
Pre-trained weights (7 billion parameters)
    ↓ train on your dataset
All 7 billion parameters get updated
    ↓
New model saved (full 7B copy)
```

**Problem**: You need a full copy of the model for each task. 7B parameters × 4 bytes = 28GB per model.

In [ ]:
# Simulate full fine-tuning
# A simple model with weights we'll fine-tune

class SimpleModel:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.3
        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.3
        self.total_params = input_dim * hidden_dim + hidden_dim * output_dim
    
    def forward(self, x):
        self.h = np.maximum(0, x @ self.W1)  # ReLU
        return self.h @ self.W2
    
    def train_step(self, x, target, lr=0.01):
        pred = self.forward(x)
        error = pred - target
        
        # Gradient descent on ALL parameters
        grad_W2 = self.h.T @ error
        grad_W1 = x.T @ ((error @ self.W2.T) * (self.h > 0))
        
        self.W1 -= lr * grad_W1
        self.W2 -= lr * grad_W2
        return np.mean(error ** 2)

# Pre-train on general data
model = SimpleModel(4, 8, 2)
print("Full Fine-Tuning Demo:\n")
print(f"  Model has {model.total_params} parameters")
print(f"  (Real LLMs: 7,000,000,000+ parameters)\n")

# Simulate pre-training (general knowledge)
print("  Phase 1: Pre-training (general data)")
general_data = np.random.randn(100, 4)
general_targets = np.random.randn(100, 2)
for epoch in range(50):
    loss = model.train_step(general_data, general_targets)
print(f"    Final loss: {loss:.4f}")
print(f"    Parameters updated: ALL {model.total_params}")

# Save pre-trained weights
pretrained_W1 = model.W1.copy()
pretrained_W2 = model.W2.copy()

# Fine-tune on specific task
print("\n  Phase 2: Fine-tuning (specific task data)")
task_data = np.random.randn(20, 4)  # much less data
task_targets = np.random.randn(20, 2)
for epoch in range(20):
    loss = model.train_step(task_data, task_targets, lr=0.001)  # smaller learning rate!
print(f"    Final loss: {loss:.4f}")
print(f"    Parameters updated: ALL {model.total_params}")

# Show how much weights changed
w1_change = np.mean(np.abs(model.W1 - pretrained_W1))
w2_change = np.mean(np.abs(model.W2 - pretrained_W2))
print(f"\n  Average weight change: W1={w1_change:.4f}, W2={w2_change:.4f}")
print(f"  (Small changes — most knowledge preserved!)")
print(f"\n  Key: use SMALLER learning rate for fine-tuning")
print(f"  Too large → forget pre-trained knowledge (catastrophic forgetting)")

---
## Catastrophic Forgetting

The biggest risk in fine-tuning: the model **forgets** what it learned during pre-training.

```
Before fine-tune: knows math, history, science, coding
After fine-tune on medical data: great at medicine, forgot math!
```

Solutions:
1. **Small learning rate** — gentle updates, less forgetting
2. **Few epochs** — don't over-train on the new data
3. **Mix in general data** — keep some original training data
4. **LoRA** — only change a tiny fraction of parameters

In [ ]:
# Demonstrate catastrophic forgetting
print("Catastrophic Forgetting Demo:\n")

# Pre-train model to do two tasks
model = SimpleModel(4, 16, 2)

# Task A data (e.g., "general knowledge")
task_a_x = np.random.randn(50, 4)
task_a_y = np.sin(task_a_x[:, :2])  # some pattern

# Task B data (e.g., "specific domain")
task_b_x = np.random.randn(50, 4)
task_b_y = np.cos(task_b_x[:, :2])  # different pattern

# Train on Task A first
for _ in range(100):
    model.train_step(task_a_x, task_a_y, lr=0.01)
loss_a_before = np.mean((model.forward(task_a_x) - task_a_y) ** 2)

print(f"  After learning Task A:")
print(f"    Task A loss: {loss_a_before:.4f} (good!)")

# Now fine-tune AGGRESSIVELY on Task B (high LR, many epochs)
print(f"\n  Fine-tuning on Task B (aggressive: high LR)...")
for _ in range(100):
    model.train_step(task_b_x, task_b_y, lr=0.03)  # high learning rate!

loss_a_after = np.mean((model.forward(task_a_x) - task_a_y) ** 2)
loss_b_after = np.mean((model.forward(task_b_x) - task_b_y) ** 2)

print(f"\n  After aggressive fine-tuning on Task B:")
print(f"    Task A loss: {loss_a_before:.4f} → {loss_a_after:.4f}  {'← FORGOT!' if loss_a_after > loss_a_before * 2 else ''}")
print(f"    Task B loss: {loss_b_after:.4f} (learned new task)")

# Now do it gently
print(f"\n  ─── Now with gentle fine-tuning (small LR, few epochs) ───")
model2 = SimpleModel(4, 16, 2)
for _ in range(100):
    model2.train_step(task_a_x, task_a_y, lr=0.01)
loss_a_before2 = np.mean((model2.forward(task_a_x) - task_a_y) ** 2)

# Gentle fine-tune
for _ in range(20):  # fewer epochs
    model2.train_step(task_b_x, task_b_y, lr=0.001)  # small LR!

loss_a_after2 = np.mean((model2.forward(task_a_x) - task_a_y) ** 2)
loss_b_after2 = np.mean((model2.forward(task_b_x) - task_b_y) ** 2)

print(f"\n  After gentle fine-tuning:")
print(f"    Task A loss: {loss_a_before2:.4f} → {loss_a_after2:.4f}  (preserved!)")
print(f"    Task B loss: {loss_b_after2:.4f} (learned some of new task)")
print(f"\n  Tradeoff: gentle = less forgetting but also less adaptation")

---
## LoRA: Low-Rank Adaptation

The breakthrough idea: instead of updating all parameters, add **small trainable matrices** alongside the frozen weights.

```
Original:     output = input × W        (W is frozen, huge)
With LoRA:    output = input × W + input × A × B   (A, B are tiny, trainable)
```

**Key insight**: the "useful change" to W during fine-tuning has **low rank** — it can be approximated by two small matrices.

```
W: 4096 × 4096 = 16,777,216 parameters (frozen)
A: 4096 × 8    =     32,768 parameters (trainable)
B: 8 × 4096    =     32,768 parameters (trainable)
                      65,536 total ← 0.4% of original!
```

In [ ]:
# LoRA from scratch
print("LoRA (Low-Rank Adaptation) — Core Idea:\n")

# Original weight matrix (frozen during fine-tuning)
d_model = 64  # real models: 4096+
W = np.random.randn(d_model, d_model) * 0.1  # pre-trained weight

# LoRA matrices (small, trainable)
rank = 4  # typically 4-64, much smaller than d_model
A = np.random.randn(d_model, rank) * 0.01  # down-project
B = np.zeros((rank, d_model))               # up-project (initialized to zero!)

print(f"  Original W: {d_model}×{d_model} = {d_model*d_model:,} parameters (FROZEN)")
print(f"  LoRA A:     {d_model}×{rank} = {d_model*rank:,} parameters (trainable)")
print(f"  LoRA B:     {rank}×{d_model} = {rank*d_model:,} parameters (trainable)")
print(f"  LoRA total: {2*d_model*rank:,} parameters")
print(f"  Reduction:  {d_model*d_model / (2*d_model*rank):.0f}x fewer trainable params!\n")

# Forward pass with LoRA
x = np.random.randn(1, d_model)  # input

original_output = x @ W           # pre-trained path
lora_output = x @ A @ B           # LoRA path (starts at 0 since B=0)
final_output = original_output + lora_output  # combined

print(f"  Forward pass:")
print(f"    original = x @ W          shape: {original_output.shape}")
print(f"    lora     = x @ A @ B      shape: {lora_output.shape}")
print(f"    output   = original + lora")
print(f"\n  At initialization (B=zeros): lora contribution = 0")
print(f"  → Model starts exactly as pre-trained!")
print(f"  → Training only updates A and B, W stays frozen")

In [ ]:
# LoRA training simulation
print("LoRA Training Simulation:\n")

class LoRALayer:
    def __init__(self, d_in, d_out, rank):
        self.W = np.random.randn(d_in, d_out) * 0.1  # frozen
        self.A = np.random.randn(d_in, rank) * 0.01
        self.B = np.zeros((rank, d_out))
        self.rank = rank
    
    def forward(self, x):
        self.x = x
        self.base = x @ self.W       # frozen path
        self.lora = x @ self.A @ self.B  # trainable path
        return self.base + self.lora
    
    def backward(self, grad_output, lr=0.01):
        # Only update A and B, NOT W!
        grad_B = (self.x @ self.A).T @ grad_output
        grad_A = self.x.T @ (grad_output @ self.B.T)
        self.B -= lr * grad_B
        self.A -= lr * grad_A
        # W is NOT updated!

# Train LoRA to learn a target transformation
layer = LoRALayer(16, 16, rank=4)
target_transform = np.random.randn(16, 16) * 0.05  # what we want to learn

losses = []
for step in range(200):
    x = np.random.randn(8, 16)
    target = x @ (layer.W + target_transform)  # target = original + small change
    
    pred = layer.forward(x)
    error = pred - target
    loss = np.mean(error ** 2)
    losses.append(loss)
    
    layer.backward(error / len(x), lr=0.05)

# Show training progress
print(f"  Training LoRA (rank={layer.rank}) to approximate a weight change:\n")
checkpoints = [0, 10, 50, 100, 199]
for i in checkpoints:
    bar = '█' * int(max(0, 30 - losses[i] * 100))
    print(f"    Step {i:>3}: loss = {losses[i]:.4f}  {bar}")

print(f"\n  Final LoRA contribution (A @ B) approximates the target change.")
actual_change = layer.A @ layer.B
approx_error = np.mean((actual_change - target_transform) ** 2)
print(f"  Approximation error: {approx_error:.6f}")
print(f"\n  W was NEVER modified — we can swap LoRA adapters without touching W!")

---
## LoRA Advantages: Swap Adapters Like Plugins

Since the base model W is frozen, you can:

```
Base Model (7B params, stored once)
    + Medical LoRA (20MB)  → Medical expert
    + Legal LoRA (20MB)   → Legal expert  
    + Code LoRA (20MB)    → Coding expert
```

Instead of 3 full copies (84GB), you store 1 base + 3 adapters (28GB + 60MB).

In [ ]:
# Multiple LoRA adapters
print("LoRA Adapter Swapping:\n")

# Base model (shared, frozen)
base_params = 7_000_000_000
base_size_gb = base_params * 2 / 1e9  # float16

# LoRA adapters (tiny)
rank = 16
n_layers = 32
d_model = 4096
lora_params_per_layer = 2 * d_model * rank  # A and B
# Applied to Q, K, V, and output projections
lora_params_total = lora_params_per_layer * 4 * n_layers
lora_size_mb = lora_params_total * 2 / 1e6  # float16

print(f"  Base model: {base_params/1e9:.0f}B params = {base_size_gb:.0f}GB (stored once)")
print(f"  Each LoRA:  {lora_params_total/1e6:.1f}M params = {lora_size_mb:.0f}MB")
print(f"  Ratio:      {lora_params_total/base_params*100:.2f}% of base model\n")

adapters = ["Medical", "Legal", "Code", "Finance", "Creative"]
print(f"  Storage comparison (5 specialized models):\n")
print(f"    Full fine-tune: 5 × {base_size_gb:.0f}GB = {5*base_size_gb:.0f}GB")
print(f"    LoRA approach:  1 × {base_size_gb:.0f}GB + 5 × {lora_size_mb:.0f}MB = {base_size_gb + 5*lora_size_mb/1000:.1f}GB")
print(f"    Savings:        {5*base_size_gb / (base_size_gb + 5*lora_size_mb/1000):.0f}x less storage!\n")

print(f"  Switching tasks at runtime:")
for adapter in adapters:
    print(f"    model.load_adapter('{adapter.lower()}_lora.bin')  # {lora_size_mb:.0f}MB swap")
print(f"\n  No need to reload the 14GB base model — just swap the adapter!")

---
## QLoRA: LoRA + Quantization

**QLoRA** = Quantized LoRA. Compress the frozen base model to 4-bit, then train LoRA on top.

```
Full model:       7B × 16 bits = 14 GB VRAM needed
Quantized (4-bit): 7B × 4 bits = 3.5 GB VRAM needed
```

This lets you fine-tune a 7B model on a single consumer GPU (e.g., RTX 3090 with 24GB).

In [ ]:
# Quantization concept
print("Quantization: Reducing Precision to Save Memory\n")

# Original weight (32-bit float)
weight = np.float32(0.23456789)

# Simulate different precisions
print(f"  Original (FP32):  {weight:.8f}  — 32 bits per number")
print(f"  Half (FP16):      {np.float16(weight):.4f}        — 16 bits per number")

# Simulate 8-bit quantization
# Map [-1, 1] range to [0, 255]
w_8bit = int((weight + 1) / 2 * 255)
w_recovered = (w_8bit / 255) * 2 - 1
print(f"  INT8:             {w_recovered:.4f}        — 8 bits per number")

# Simulate 4-bit quantization
w_4bit = int((weight + 1) / 2 * 15)
w_recovered_4 = (w_4bit / 15) * 2 - 1
print(f"  INT4:             {w_recovered_4:.4f}        — 4 bits per number")

print(f"\n  Memory for 7B parameter model:")
print(f"    FP32: {7e9 * 4 / 1e9:.0f} GB")
print(f"    FP16: {7e9 * 2 / 1e9:.0f} GB")
print(f"    INT8: {7e9 * 1 / 1e9:.0f} GB")
print(f"    INT4: {7e9 * 0.5 / 1e9:.1f} GB  ← fits on a gaming GPU!")

print(f"\n  QLoRA: base model in INT4 (3.5GB) + LoRA adapters in FP16")
print(f"  Total: ~4-5 GB for fine-tuning a 7B model!")

---
## Fine-Tuning Data Formats

Different tasks need different data formats:

```
Instruction tuning:  {instruction, input, output}
Chat fine-tuning:    {messages: [{role, content}, ...]}
Classification:      {text, label}
Completion:          {prompt, completion}
```

In [ ]:
# Common fine-tuning data formats
print("Fine-Tuning Data Formats:\n")

# Format 1: Instruction tuning (Alpaca style)
print("1. Instruction Tuning (Alpaca format):")
alpaca_example = {
    "instruction": "Summarize the following text in one sentence.",
    "input": "The stock market fell 3% today due to concerns about...",
    "output": "Stock markets declined 3% amid economic concerns."
}
for k, v in alpaca_example.items():
    print(f"    {k}: \"{v}\"")

# Format 2: Chat format
print("\n2. Chat Format (OpenAI/Claude style):")
chat_example = {
    "messages": [
        {"role": "system", "content": "You are a medical assistant."},
        {"role": "user", "content": "What causes headaches?"},
        {"role": "assistant", "content": "Common causes include tension, dehydration..."}
    ]
}
for msg in chat_example["messages"]:
    print(f"    {msg['role']}: \"{msg['content']}\"")

# Format 3: Preference pairs (for DPO)
print("\n3. Preference Pairs (for DPO fine-tuning):")
pref_example = {
    "prompt": "Explain quantum computing.",
    "chosen": "Quantum computing uses qubits that can be in superposition...",
    "rejected": "Quantum computers are really fast computers that..."
}
for k, v in pref_example.items():
    print(f"    {k}: \"{v}\"")

print("\n  Typical dataset sizes:")
print("    Instruction tuning: 10K-100K examples")
print("    Chat fine-tuning:   1K-50K conversations")
print("    DPO:                1K-10K preference pairs")

---
## The Fine-Tuning Training Loop

Fine-tuning uses the same training loop as pre-training (predict next token), but:
1. **Only on your data** (not the internet)
2. **Smaller learning rate** (don't overwrite pre-trained knowledge)
3. **Fewer epochs** (1-3 is typical, vs. 1 epoch for pre-training)
4. **Loss only on outputs** (mask the prompt, only learn from the response)

In [ ]:
# Training loss masking
print("Loss Masking: Only Learn From the Response\n")

# Full conversation
tokens =   ["[User]", "What", "is", "Python", "?", "[Assistant]", "Python", "is", "a", "language", "."]
# Loss mask: 0 = ignore, 1 = compute loss
loss_mask = [0,        0,      0,    0,        0,   0,             1,        1,    1,   1,          1]

print(f"  Token:     ", end="")
for t in tokens:
    print(f"{t:>11}", end="")
print()

print(f"  Loss mask: ", end="")
for m in loss_mask:
    marker = "LEARN" if m else "skip"
    print(f"{marker:>11}", end="")
print()

print(f"\n  Why? We don't want the model to 'learn' to generate user prompts.")
print(f"  We only want it to learn how to RESPOND to prompts.")
print(f"  The prompt tokens still provide context through attention,")
print(f"  but they don't contribute to the loss/gradient.")

print(f"\n\nTypical Fine-Tuning Hyperparameters:")
print(f"  ┌────────────────────┬──────────────────┬──────────────────┐")
print(f"  │ Parameter          │ Pre-training     │ Fine-tuning      │")
print(f"  ├────────────────────┼──────────────────┼──────────────────┤")
print(f"  │ Learning rate      │ 1e-4 to 3e-4    │ 1e-5 to 5e-5    │")
print(f"  │ Epochs             │ 1 (huge data)    │ 1-3 (small data) │")
print(f"  │ Batch size         │ 1M+ tokens       │ 32-128 examples  │")
print(f"  │ Warmup steps       │ 2000             │ 100-500          │")
print(f"  │ Weight decay       │ 0.1              │ 0.01-0.1        │")
print(f"  └────────────────────┴──────────────────┴──────────────────┘")

---
## When to Use What

```
"I want my model to..."          → Best approach
─────────────────────────────────────────────────
Follow a specific format          → Few-shot prompting (free)
Know about my company's docs      → RAG (Ch 18)
Be an expert in one domain        → LoRA fine-tuning
Have a specific personality/style  → SFT + LoRA
Be safe and aligned               → RLHF/DPO (Ch 14)
Do everything from scratch        → Full pre-training ($$$)
```

In [ ]:
# Decision tree for choosing approach
print("Decision Tree: How to Adapt an LLM\n")
print("  Do you need the model to know new FACTS?")
print("  │")
print("  ├── Yes → Do the facts change frequently?")
print("  │         │")
print("  │         ├── Yes → Use RAG (retrieve at query time)")
print("  │         │")
print("  │         └── No → Fine-tune (bake knowledge in)")
print("  │")
print("  └── No → Do you need a different STYLE/FORMAT?")
print("           │")
print("           ├── Simple format → Prompt engineering (system prompt)")
print("           │")
print("           └── Complex behavior → Fine-tune (SFT/LoRA)")
print()
print("\n  Common combinations:")
print("    • RAG + Fine-tuning: model knows how to USE retrieved docs")
print("    • LoRA + DPO: domain expert that's also safe/helpful")
print("    • Prompt + RAG: cheapest effective approach for most cases")

print("\n\n  Real-world fine-tuning examples:")
examples = [
    ("Code Llama",     "LLaMA",    "Fine-tuned on code",          "LoRA + full FT"),
    ("BioMistral",     "Mistral",  "Fine-tuned on medical papers", "Full FT"),
    ("WizardMath",     "LLaMA",    "Fine-tuned on math problems",  "SFT + RL"),
    ("Gorilla",        "LLaMA",    "Fine-tuned on API docs",       "SFT"),
]
print(f"    {'Model':<14} {'Base':<10} {'Data':<30} {'Method'}")
print(f"    {'─'*14} {'─'*10} {'─'*30} {'─'*12}")
for name, base, data, method in examples:
    print(f"    {name:<14} {base:<10} {data:<30} {method}")

---
## Putting It All Together: The Modern LLM Pipeline

```
Step 1: Pre-train base model (Ch 13)
    → Raw text prediction, trillions of tokens, $millions

Step 2: SFT / Instruction Fine-tune (Ch 14)
    → Follows instructions, 10K-100K examples

Step 3: RLHF / DPO (Ch 14)
    → Safe, helpful, honest

Step 4: Domain LoRA (this chapter)
    → Specialized for your use case

Step 5: RAG at inference time (Ch 18)
    → Access to current/private information
```

In [ ]:
# Full pipeline summary
print("The Complete LLM Training Pipeline:\n")
print("  ┌─────────────────────────────────────────────────────────────┐")
print("  │ Stage 1: PRE-TRAINING                                      │")
print("  │   Data:   Internet text (trillions of tokens)              │")
print("  │   Cost:   $1M - $100M                                      │")
print("  │   Result: Base model (predicts next token)                  │")
print("  │   Who:    OpenAI, Anthropic, Meta, Google                   │")
print("  └─────────────────────────┬───────────────────────────────────┘")
print("                            ↓")
print("  ┌─────────────────────────────────────────────────────────────┐")
print("  │ Stage 2: INSTRUCTION FINE-TUNING (SFT)                     │")
print("  │   Data:   (instruction, response) pairs — 10K-100K         │")
print("  │   Cost:   $100 - $10K                                      │")
print("  │   Result: Model that follows instructions                   │")
print("  └─────────────────────────┬───────────────────────────────────┘")
print("                            ↓")
print("  ┌─────────────────────────────────────────────────────────────┐")
print("  │ Stage 3: ALIGNMENT (RLHF / DPO)                            │")
print("  │   Data:   Human preferences — 1K-10K pairs                 │")
print("  │   Cost:   $1K - $100K (human labelers expensive)           │")
print("  │   Result: Safe, helpful, honest assistant                   │")
print("  └─────────────────────────┬───────────────────────────────────┘")
print("                            ↓")
print("  ┌─────────────────────────────────────────────────────────────┐")
print("  │ Stage 4: DOMAIN ADAPTATION (LoRA / QLoRA)                   │")
print("  │   Data:   Domain-specific examples — 1K-10K                │")
print("  │   Cost:   $10 - $100                                       │")
print("  │   Result: Domain expert (medical, legal, code, etc.)        │")
print("  │   Who:    Anyone with a GPU!                                │")
print("  └─────────────────────────────────────────────────────────────┘")
print()
print("  Each stage builds on the previous one.")
print("  Most users only need Stage 4 (or just RAG + prompting).")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **Full fine-tuning** | Update all parameters — expensive but powerful |
| **LoRA** | Add small matrices (A×B), freeze original weights |
| **QLoRA** | LoRA + 4-bit quantized base model → fits on consumer GPU |
| **Catastrophic forgetting** | Over-training erases pre-trained knowledge |
| **Loss masking** | Only compute loss on the response, not the prompt |
| **Adapter swapping** | One base model + many small LoRA adapters |
| **When to fine-tune** | Need new behavior/style that prompting can't achieve |

**Part 3 Complete!** Next: Part 4 (Applied AI) — data pipelines, evaluation, RAG, deployment